In [1]:
import requests
import json
import pandas as pd
import numpy as np
import os

In [2]:
df_user = pd.read_csv("data/usuarios.csv")
df_user

,id_user,nombre,apellido,email,password_hash,tlf,municipality_id,sexo,age,role,created_at,updated_at
0,1,Ander,Martin Alonso,ander.martin1075@euskador.eus,0d6e670ed0489d3dd6fe00c0113f7464203a0240c5e0b4...,+34 98266521,20055,hombre,47,user,2026-03-16 10:22:00,2026-03-16 10:22:00
1,2,Daniel,Agirre Ruiz,daniel.agirre849@euskador.eus,efc20d1042848eb9674b1a9a339abeeb96cc98935f488c...,+34 97389339,48015,hombre,58,user,2026-03-20 13:43:00,2026-03-20 13:43:00
2,3,Joseba,Garate Moreno,joseba.garate897@euskador.eus,dc00655e589390cd9ed0ebf7f0a33176829bafa0ae9ce6...,+34 69278928,48027,hombre,45,user,2026-03-18 06:39:00,2026-03-18 06:39:00
3,4,Carmen,González Goikoetxea,carmen.gonzalez225@euskador.eus,a7d8680c5e5641800b0e47c63d32971def5487a274f483...,+34 71486009,48015,mujer,65,user,2026-03-07 11:31:00,2026-03-07 11:31:00
4,5,Itziar,Larrañaga Alonso,itziar.larranaga419@euskador.eus,13df7d7f2c511b6bc07133adfa895e57e089fe45e08c37...,+34 76041284,48011,mujer,60,user,2026-03-28 04:40:00,2026-03-28 04:40:00
...,...,...,...,...,...,...,...,...,...,...,...,...
1995,1996,Manuel,Mendizabal Arregi,manuel.mendizabal204@euskador.eus,383f992cd7c848cc4f053cac521f03502541ac4a50898c...,+34 74645676,48011,hombre,47,user,2026-03-09 13:59:00,2026-03-09 13:59:00
1996,1997,Imanol,Muñoz Zabaleta,imanol.munoz123@euskador.eus,95392d1f2ad36f998695e78e63a19d33740b138711d83f...,+34 98719044,48003,hombre,54,user,2026-03-28 08:13:00,2026-03-28 08:13:00
1997,1998,Iñaki,Goikoetxea Barandiaran,inaki.goikoetxea1517@euskador.eus,7c93c49fc521604ae3f6c9e7144a4c4edfbc0ef503a9a7...,+34 65387151,20055,hombre,63,user,2026-03-28 15:54:00,2026-03-28 15:54:00
1998,1999,Miren,Goikoetxea Basabe,miren.goikoetxea1707@euskador.eus,bd9ac7b1d4dd6b10cdcd4870a792cda2b7696e349f42b0...,+34 73993656,1043,mujer,58,user,2026-03-18 09:40:00,2026-03-18 09:40:00


In [3]:
df_user.drop(columns=["nombre", "apellido", "email", "password_hash", "tlf", "role", "created_at", "updated_at"], inplace=True)

In [4]:
df_user

,id_user,municipality_id,sexo,age
0,1,20055,hombre,47
1,2,48015,hombre,58
2,3,48027,hombre,45
3,4,48015,mujer,65
4,5,48011,mujer,60
...,...,...,...,...
1995,1996,48011,hombre,47
1996,1997,48003,hombre,54
1997,1998,20055,hombre,63
1998,1999,1043,mujer,58


In [5]:
#listamos las columnas de df_user
print("df_user columns:", df_user.columns.tolist())

df_user columns: ['id_user', 'municipality_id', 'sexo', 'age']


In [6]:
df_intereses = pd.read_csv("data/intereses.csv")
df_intereses

,id_interes,nombre,father_id,level
0,1,Eventos,NaN,0
1,2,Gastronomía,NaN,0
2,3,Puntos de interés,NaN,0
3,4,Concierto y Festival,1.0,1
4,5,Fiestas y Feria,1.0,1
5,6,Teatro,1.0,1
6,7,Danza,1.0,1
7,8,"Conferencia, Eventos/jornadas, Presentación",1.0,1
8,9,Cine y audiovisuales,1.0,1
9,10,Bertsolarismo,1.0,1


In [7]:
df_user_interes = pd.read_csv("data/intereses_usuarios.csv")
df_user_interes

,id_user,id_interes
0,1,14
1,1,17
2,1,20
3,1,27
4,1,29
...,...,...
9167,2000,8
9168,2000,14
9169,2000,17
9170,2000,20


In [8]:
# unimos df_user_interes con df_user (por id_user) y con df_intereses (por id_interes)
# luego generamos un one-hot encoding con los intereses como columnas conservando las columnas de df_user
# no perdemos usuarios sin intereses: partimos de todos los usuarios y rellenamos con 0
user_cols = ["id_user", "municipality_id", "sexo", "age"]

base_users = df_user[user_cols].drop_duplicates().copy()

intereses_por_usuario = (
    df_user_interes
    .merge(df_intereses[["id_interes", "nombre"]], on="id_interes", how="left")
    .drop(columns=["id_interes"])
)

df_user_total = (
    base_users
    .merge(intereses_por_usuario, on="id_user", how="left")
)

df_user_total = (
    df_user_total
    .pivot_table(index=user_cols, columns="nombre", aggfunc="size", fill_value=0)
    .reset_index()
)

df_user_total.columns.name = None

# aseguramos que también estén los usuarios sin intereses
all_users_index = base_users.set_index(user_cols).index
current_index = df_user_total.set_index(user_cols).index
df_user_total = (
    df_user_total
    .set_index(user_cols)
    .reindex(all_users_index, fill_value=0)
    .reset_index()
)

# por si alguna columna de interés queda como float tras el reindex, la pasamos a entero
interest_cols = [c for c in df_user_total.columns if c not in user_cols]
df_user_total[interest_cols] = df_user_total[interest_cols].astype(int)

df_user_total


,id_user,municipality_id,sexo,age,Agricultura ecológica,Arte,Asador,Bertsolarismo,Bodegas,Ciencias naturales,...,Fiestas y Feria,Formación,Gourmet,Hostelería,Museos: Historia,Patrimonio cultural,Queserías,Restaurante,Sidrería,Teatro
0,1,20055,hombre,47,0,1,0,0,1,0,...,0,0,0,1,0,1,0,1,0,0
1,2,48015,hombre,58,0,1,0,0,1,0,...,0,0,0,1,0,1,0,1,0,0
2,3,48027,hombre,45,0,1,1,0,0,0,...,0,0,0,1,0,1,0,1,0,0
3,4,48015,mujer,65,0,1,1,0,1,0,...,0,0,0,1,0,1,0,1,0,1
4,5,48011,mujer,60,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1995,1996,48011,hombre,47,0,1,1,0,0,0,...,0,0,0,1,0,0,0,1,0,0
1996,1997,48003,hombre,54,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1997,1998,20055,hombre,63,0,1,0,0,1,0,...,0,0,0,1,0,1,0,0,0,1
1998,1999,1043,mujer,58,0,0,1,0,1,0,...,0,0,0,1,0,0,0,0,0,1


In [9]:
# la columna df_user_total hay que editarla para quedanos con lo 2 primeros digitos para obtener la provincia
df_user_total["municipality_id"] = df_user_total["municipality_id"].astype(str).str[:2]
df_user_total

,id_user,municipality_id,sexo,age,Agricultura ecológica,Arte,Asador,Bertsolarismo,Bodegas,Ciencias naturales,...,Fiestas y Feria,Formación,Gourmet,Hostelería,Museos: Historia,Patrimonio cultural,Queserías,Restaurante,Sidrería,Teatro
0,1,20,hombre,47,0,1,0,0,1,0,...,0,0,0,1,0,1,0,1,0,0
1,2,48,hombre,58,0,1,0,0,1,0,...,0,0,0,1,0,1,0,1,0,0
2,3,48,hombre,45,0,1,1,0,0,0,...,0,0,0,1,0,1,0,1,0,0
3,4,48,mujer,65,0,1,1,0,1,0,...,0,0,0,1,0,1,0,1,0,1
4,5,48,mujer,60,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1995,1996,48,hombre,47,0,1,1,0,0,0,...,0,0,0,1,0,0,0,1,0,0
1996,1997,48,hombre,54,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1997,1998,20,hombre,63,0,1,0,0,1,0,...,0,0,0,1,0,1,0,0,0,1
1998,1999,10,mujer,58,0,0,1,0,1,0,...,0,0,0,1,0,0,0,0,0,1


In [10]:
# ahora hacemos un one-hot de municipality_id y de sexo
df_user_total = pd.get_dummies(df_user_total, columns=["municipality_id", "sexo"], prefix=["municipality", "sexo"])
df_user_total

,id_user,age,Agricultura ecológica,Arte,Asador,Bertsolarismo,Bodegas,Ciencias naturales,Cine y audiovisuales,Concierto y Festival,...,Queserías,Restaurante,Sidrería,Teatro,municipality_10,municipality_20,municipality_48,sexo_hombre,sexo_mujer,sexo_otro
0,1,47,0,1,0,0,1,0,0,0,...,0,1,0,0,False,True,False,True,False,False
1,2,58,0,1,0,0,1,0,0,0,...,0,1,0,0,False,False,True,True,False,False
2,3,45,0,1,1,0,0,0,0,0,...,0,1,0,0,False,False,True,True,False,False
3,4,65,0,1,1,0,1,0,0,0,...,0,1,0,1,False,False,True,False,True,False
4,5,60,0,0,0,0,0,0,0,0,...,0,0,0,0,False,False,True,False,True,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1995,1996,47,0,1,1,0,0,0,0,0,...,0,1,0,0,False,False,True,True,False,False
1996,1997,54,0,0,0,0,0,0,0,0,...,0,0,0,0,False,False,True,True,False,False
1997,1998,63,0,1,0,0,1,0,0,0,...,0,0,0,1,False,True,False,True,False,False
1998,1999,58,0,0,1,0,1,0,0,0,...,0,0,0,1,True,False,False,False,True,False


In [11]:
df_user_total.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 34 columns):
 #   Column                                       Non-Null Count  Dtype
---  ------                                       --------------  -----
 0   id_user                                      2000 non-null   int64
 1   age                                          2000 non-null   int64
 2   Agricultura ecológica                        2000 non-null   int32
 3   Arte                                         2000 non-null   int32
 4   Asador                                       2000 non-null   int32
 5   Bertsolarismo                                2000 non-null   int32
 6   Bodegas                                      2000 non-null   int32
 7   Ciencias naturales                           2000 non-null   int32
 8   Cine y audiovisuales                         2000 non-null   int32
 9   Concierto y Festival                         2000 non-null   int32
 10  Concurso                

In [12]:
df_user_total.to_csv("ML/data/user_total.csv", index=False)

# GASTRONOMIA

In [13]:
df_gastro = pd.read_csv("data/gastronomia.csv")
df_gastro

,Unnamed: 0,Nombre,Descripcion,Dirección,Municipio,Provincia,Entorno,Email,Teléfono,WEB,...,Calidad,Tipo de lugar,Nivel precio,lat,lon,valoracion,num_resenas,url_imagen,Active,Patrocinado
0,0,Agorregi,"El restaurante Agorregi, ubicado en el barrio ...","Portuetxe K., 14, 20018 Donostia / San Sebasti...",San Sebastián,Guipúzcoa,"Costa Vasca,San Sebastián",agorregi@agorregi.com,943 22 43 28,https://agorregi.com/,...,0,Restaurante,Moderado,43.302405,-2.011846,4.6,571.0,https://places.googleapis.com/v1/places/ChIJFT...,1,0
1,1,Aizian,Este moderno y acogedor restaurante fue diseña...,"Leizaola Lehendakariaren Kalea, 29, Abando, 48...",Bilbao,Vizcaya,Bilbao,aizian@restaurante-aizian.com,944 28 00 39,https://www.restaurante-aizian.com/,...,0,Restaurante,Moderado,43.267519,-2.941807,4.7,435.0,https://places.googleapis.com/v1/places/ChIJde...,1,0
2,2,Akelarre,Pedro Subijana desarrolla desde 1970 una cocin...,"Padre Orkolaga Ibilbidea, 56, 20008 Donostia /...",San Sebastián,Guipúzcoa,San Sebastián,restaurante@akelarre.net,943 31 12 09,https://www.akelarre.net,...,1,Restaurante,Muy caro,43.307750,-2.043135,4.5,1925.0,https://places.googleapis.com/v1/places/ChIJXd...,1,0
3,3,Alameda,La tercera generación es quien está a cargo de...,"Minasoroeta Kalea, 1, 20280, Gipuzkoa, Spain",Hondarribia,Guipúzcoa,Costa Vasca,info@restaurantealameda.net,943 64 27 89,https://restaurantealameda.net/,...,1,Restaurante,Caro,43.361242,-1.792691,4.6,1098.0,https://places.googleapis.com/v1/places/ChIJfb...,1,0
4,4,Almiketxu,Es un caserío típico vasco y está regentado po...,"Almike Auzoa, 8, 48370 Bermeo, Bizkaia, Spain",Bermeo,Vizcaya,Costa Vasca,almiketxu@gmail.com,946 88 09 25,https://almiketxu.com/,...,1,Restaurante,Caro,43.408905,-2.734229,4.6,1924.0,https://places.googleapis.com/v1/places/ChIJZ5...,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
485,765,Quesería Soloitza,Quesería Soloitza,"Barrio Esq. Abajo, 01476 Arespalditza / Respal...",Ayala,Álava,Montes y Valles vascos,info@soloitza.com,635 71 84 75,https://soloitza.com/,...,0,tienda,Moderado,43.092827,-3.048182,5.0,45.0,https://places.googleapis.com/v1/places/ChIJmY...,1,0
486,767,Larrañaga Gozotegia,Repostería tradicional. De su obrador salen l...,"San Pedro Kalea, 9, 20570 Bergara, Gipuzkoa, S...",Bergara,Guipúzcoa,Montes y Valles vascos,NaN,943 76 10 51,NaN,...,1,tienda,Moderado,43.116464,-2.412943,4.6,70.0,https://places.googleapis.com/v1/places/ChIJLe...,1,0
487,772,Pastelería Sosoaga,"La pastelería López Sosoaga, abierta en 1868 p...","Diputación, 9, 01005 Vitoria-Gasteiz, Araba, S...",Vitoria,Álava,Vitoria-Gasteiz,NaN,945 25 85 73,http://www.sosoaga.com,...,1,tienda,Moderado,42.842886,-2.668233,4.3,186.0,https://places.googleapis.com/v1/places/ChIJ1w...,1,0
488,773,Pastelería La Peña Dulce,La Peña Dulce es una de las pastelerías de may...,"Hedegile Kalea, 124, 01001 Vitoria-Gasteiz, Ar...",Vitoria,Álava,Vitoria-Gasteiz,NaN,945 13 26 37,NaN,...,1,tienda,Moderado,42.851866,-2.673002,4.2,106.0,https://places.googleapis.com/v1/places/ChIJVc...,1,0


In [14]:
#Eliminamos las columnas "Nombre", "Descripción", "Dirección", "Municipio", "Emiail", "Teléfono", "WEB", "URL amigable", "Patrocinado", "updated_at"
df_gastro = df_gastro.drop(columns=["Nombre", "Descripcion", "Dirección", "Municipio", "Email", "Teléfono", "WEB", "URL amigable", "url_imagen", "lat", "lon", "Patrocinado", "Active"])
df_gastro

,Unnamed: 0,Provincia,Entorno,Calidad,Tipo de lugar,Nivel precio,valoracion,num_resenas
0,0,Guipúzcoa,"Costa Vasca,San Sebastián",0,Restaurante,Moderado,4.6,571.0
1,1,Vizcaya,Bilbao,0,Restaurante,Moderado,4.7,435.0
2,2,Guipúzcoa,San Sebastián,1,Restaurante,Muy caro,4.5,1925.0
3,3,Guipúzcoa,Costa Vasca,1,Restaurante,Caro,4.6,1098.0
4,4,Vizcaya,Costa Vasca,1,Restaurante,Caro,4.6,1924.0
...,...,...,...,...,...,...,...,...
485,765,Álava,Montes y Valles vascos,0,tienda,Moderado,5.0,45.0
486,767,Guipúzcoa,Montes y Valles vascos,1,tienda,Moderado,4.6,70.0
487,772,Álava,Vitoria-Gasteiz,1,tienda,Moderado,4.3,186.0
488,773,Álava,Vitoria-Gasteiz,1,tienda,Moderado,4.2,106.0


In [15]:
#renombramos la columana "Unnamed: 0" como gastro_id
df_gastro.rename(columns={"Unnamed: 0": "gastro_id"}, inplace=True)

In [16]:
# listamos y contamos los distintos valores de la columna "Entorno"
print("Valores únicos en la columna 'Entorno':", df_gastro["Entorno"].unique())
print("Conteo de cada valor en la columna 'Entorno':\n", df_gastro["Entorno"].value_counts())

Valores únicos en la columna 'Entorno': ['Costa Vasca,San Sebastián' 'Bilbao' 'San Sebastián' 'Costa Vasca'
 'Vitoria-Gasteiz' 'Montes y Valles vascos'
 'Costa Vasca,Montes y Valles vascos' 'Rioja Alavesa' 'Costa Vasca,Bilbao'
 'Montes y Valles vascos,San Sebastián']
Conteo de cada valor en la columna 'Entorno':
 Entorno
Montes y Valles vascos                  191
Costa Vasca                             104
Rioja Alavesa                            80
Bilbao                                   33
Costa Vasca,San Sebastián                28
Vitoria-Gasteiz                          18
Montes y Valles vascos,San Sebastián     17
San Sebastián                            14
Costa Vasca,Montes y Valles vascos        4
Costa Vasca,Bilbao                        1
Name: count, dtype: int64


In [17]:
# comprobación: no deben quedar columnas de Entorno con coma
entorno_cols = [c for c in df_gastro.columns if c.startswith("Entorno_")]
con_coma = [c for c in entorno_cols if "," in c]

print("Total columnas Entorno:", len(entorno_cols))
print("Columnas Entorno con coma:", con_coma)
print("Suma por columna Entorno:\n", df_gastro[entorno_cols].sum().sort_values(ascending=False))


Total columnas Entorno: 0
Columnas Entorno con coma: []
Suma por columna Entorno:
 Series([], dtype: float64)


In [18]:
df_gastro.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 490 entries, 0 to 489
Data columns (total 8 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   gastro_id      490 non-null    int64  
 1   Provincia      490 non-null    object 
 2   Entorno        490 non-null    object 
 3   Calidad        490 non-null    int64  
 4   Tipo de lugar  490 non-null    object 
 5   Nivel precio   490 non-null    object 
 6   valoracion     490 non-null    float64
 7   num_resenas    490 non-null    float64
dtypes: float64(2), int64(2), object(4)
memory usage: 30.8+ KB


In [19]:
df_qualifications = pd.read_csv("data/gastronomy_qualifications.csv")
df_qualifications


,Michelin,Repsol,Denominación de Origen,Agricultura Ecológica,Euskal Baserri,Eusko Label
0,0,0,0,0,0,0
1,0,0,0,0,0,0
2,1,1,0,0,0,0
3,1,1,0,0,0,0
4,0,0,0,0,0,0
...,...,...,...,...,...,...
485,0,0,0,0,0,1
486,0,0,1,0,0,0
487,0,0,1,0,0,0
488,0,0,1,0,0,0


In [20]:
# creamos gastro_id en df_qualifications usando los mismos IDs de df_gastro (mismo orden fila a fila)
df_qualifications = df_qualifications.copy()
df_qualifications["gastro_id"] = df_gastro["gastro_id"].values
df_qualifications


,Michelin,Repsol,Denominación de Origen,Agricultura Ecológica,Euskal Baserri,Eusko Label,gastro_id
0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,1
2,1,1,0,0,0,0,2
3,1,1,0,0,0,0,3
4,0,0,0,0,0,0,4
...,...,...,...,...,...,...,...
485,0,0,0,0,0,1,765
486,0,0,1,0,0,0,767
487,0,0,1,0,0,0,772
488,0,0,1,0,0,0,773


In [21]:
# unimos df_gastro con df_qualifications por gastro_id
df_gastro_total = df_gastro.merge(df_qualifications, on="gastro_id", how="left")
df_gastro_total

,gastro_id,Provincia,Entorno,Calidad,Tipo de lugar,Nivel precio,valoracion,num_resenas,Michelin,Repsol,Denominación de Origen,Agricultura Ecológica,Euskal Baserri,Eusko Label
0,0,Guipúzcoa,"Costa Vasca,San Sebastián",0,Restaurante,Moderado,4.6,571.0,0,0,0,0,0,0
1,1,Vizcaya,Bilbao,0,Restaurante,Moderado,4.7,435.0,0,0,0,0,0,0
2,2,Guipúzcoa,San Sebastián,1,Restaurante,Muy caro,4.5,1925.0,1,1,0,0,0,0
3,3,Guipúzcoa,Costa Vasca,1,Restaurante,Caro,4.6,1098.0,1,1,0,0,0,0
4,4,Vizcaya,Costa Vasca,1,Restaurante,Caro,4.6,1924.0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
485,765,Álava,Montes y Valles vascos,0,tienda,Moderado,5.0,45.0,0,0,0,0,0,1
486,767,Guipúzcoa,Montes y Valles vascos,1,tienda,Moderado,4.6,70.0,0,0,1,0,0,0
487,772,Álava,Vitoria-Gasteiz,1,tienda,Moderado,4.3,186.0,0,0,1,0,0,0
488,773,Álava,Vitoria-Gasteiz,1,tienda,Moderado,4.2,106.0,0,0,1,0,0,0


In [22]:
# one-hot robusto para Entorno: separa etiquetas múltiples por coma
# si df_gastro_total ya está one-hot de una ejecución previa, lo reconstruimos desde el merge base
if "Entorno" not in df_gastro_total.columns:
    df_gastro_total = df_gastro.merge(df_qualifications, on="gastro_id", how="left")

df_gastro_total = df_gastro_total.copy()

# 1) limpiamos espacios y separamos en lista de etiquetas
entorno_series = (
    df_gastro_total["Entorno"]
    .fillna("")
    .astype(str)
    .str.split(",")
    .apply(lambda vals: [v.strip() for v in vals if v.strip()])
)

# 2) multi-hot encoding de Entorno (una columna por etiqueta)
entorno_dummies = entorno_series.str.join("|").str.get_dummies(sep="|")
entorno_dummies = entorno_dummies.add_prefix("Entorno_")

# 3) one-hot estándar para el resto de columnas categóricas
df_gastro_total = pd.get_dummies(
    df_gastro_total,
    columns=["Provincia", "Tipo de lugar", "Nivel precio"],
    prefix=["Provincia", "Tipo de lugar", "Nivel precio"]
)

# 4) quitamos Entorno original y añadimos columnas multi-hot
df_gastro_total = pd.concat(
    [df_gastro_total.drop(columns=["Entorno"]), entorno_dummies],
    axis=1
)

# verificación rápida
print("Columnas Entorno creadas:", sorted([c for c in df_gastro_total.columns if c.startswith("Entorno_")]))
print("Total columnas Entorno:", len([c for c in df_gastro_total.columns if c.startswith("Entorno_")]))


Columnas Entorno creadas: ['Entorno_Bilbao', 'Entorno_Costa Vasca', 'Entorno_Montes y Valles vascos', 'Entorno_Rioja Alavesa', 'Entorno_San Sebastián', 'Entorno_Vitoria-Gasteiz']
Total columnas Entorno: 6


In [23]:
df_gastro_total.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 490 entries, 0 to 489
Data columns (total 29 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   gastro_id                       490 non-null    int64  
 1   Calidad                         490 non-null    int64  
 2   valoracion                      490 non-null    float64
 3   num_resenas                     490 non-null    float64
 4   Michelin                        490 non-null    int64  
 5   Repsol                          490 non-null    int64  
 6   Denominación de Origen          490 non-null    int64  
 7   Agricultura Ecológica           490 non-null    int64  
 8   Euskal Baserri                  490 non-null    int64  
 9   Eusko Label                     490 non-null    int64  
 10  Provincia_Guipúzcoa             490 non-null    bool   
 11  Provincia_Vizcaya               490 non-null    bool   
 12  Provincia_Álava                 490 

In [24]:
df_gastro_total.to_csv("data/gastro_total.csv", index=False)

# VALORACIONES GASTRO

In [25]:
df_reviews = pd.read_csv("data/reviews_gastronomy.csv")
df_reviews

,id,user_id,gastro_id,puntuacion,texto,created_at
0,1,1441,0,4,Muy buena comida y ambiente agradable. Volvere...,2026-05-23 18:30:00
1,2,1326,0,4,Muy buena comida y ambiente agradable. Volvere...,2026-03-16 13:40:00
2,3,211,0,5,"Comida espectacular, servicio impecable. De lo...",2026-05-15 19:45:00
3,4,1833,0,3,"Buen ambiente, pero la comida nos dejó un poco...",2026-05-09 18:42:00
4,5,1584,0,5,Una experiencia gastronómica increíble. Todo r...,2026-05-18 20:44:00
...,...,...,...,...,...,...
15576,15577,1700,775,5,"Comida espectacular, servicio impecable. De lo...",2026-03-15 20:59:00
15577,15578,1016,775,4,Gran sitio para ir en cuadrilla o familia. Los...,2026-05-27 15:00:00
15578,15579,1033,775,4,"Todo muy rico, buena atención y raciones gener...",2026-05-04 18:38:00
15579,15580,1879,775,5,"Comida espectacular, servicio impecable. De lo...",2026-05-04 22:32:00


In [26]:
# hacemos drop de las columnas "id", "texto", "created_at", "updated_at"
df_reviews = df_reviews.drop(columns=["id", "texto", "created_at", "created_at"])
df_reviews

,user_id,gastro_id,puntuacion
0,1441,0,4
1,1326,0,4
2,211,0,5
3,1833,0,3
4,1584,0,5
...,...,...,...
15576,1700,775,5
15577,1016,775,4
15578,1033,775,4
15579,1879,775,5


In [27]:
df_reviews.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15581 entries, 0 to 15580
Data columns (total 3 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   user_id     15581 non-null  int64
 1   gastro_id   15581 non-null  int64
 2   puntuacion  15581 non-null  int64
dtypes: int64(3)
memory usage: 365.3 KB


In [28]:
# unimos por claves correctas:
# df_reviews.user_id -> df_user_total.id_user
# df_reviews.gastro_id -> df_gastro_total.gastro_id
df_reviews_total = (
    df_reviews
    .merge(df_user_total, left_on="user_id", right_on="id_user", how="left")
    .merge(df_gastro_total, on="gastro_id", how="left")
)

# comprobación rápida
print("Filas df_reviews:", len(df_reviews))
print("Filas df_reviews_total:", len(df_reviews_total))
df_reviews_total.head()


Filas df_reviews: 15581
Filas df_reviews_total: 15581


,user_id,gastro_id,puntuacion,id_user,age,Agricultura ecológica,Arte,Asador,Bertsolarismo,Bodegas,...,Tipo de lugar_tienda,Nivel precio_Caro,Nivel precio_Moderado,Nivel precio_Muy caro,Entorno_Bilbao,Entorno_Costa Vasca,Entorno_Montes y Valles vascos,Entorno_Rioja Alavesa,Entorno_San Sebastián,Entorno_Vitoria-Gasteiz
0,1441,0,4,1441,48,0,0,1,0,1,...,False,False,True,False,0,1,0,0,1,0
1,1326,0,4,1326,50,0,1,1,0,1,...,False,False,True,False,0,1,0,0,1,0
2,211,0,5,211,20,0,0,0,0,0,...,False,False,True,False,0,1,0,0,1,0
3,1833,0,3,1833,55,0,0,0,0,0,...,False,False,True,False,0,1,0,0,1,0
4,1584,0,5,1584,60,0,0,1,0,1,...,False,False,True,False,0,1,0,0,1,0


In [29]:
# Eliminamos las columnas "user_id" e "id_user" (ya tenemos toda la info de usuario en df_reviews_total) y "gastro_id" (ya tenemos toda la info de gastro en df_reviews_total)
df_reviews_total = df_reviews_total.drop(columns=["user_id", "id_user", "gastro_id"])

In [30]:
df_reviews_total.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15581 entries, 0 to 15580
Data columns (total 62 columns):
 #   Column                                       Non-Null Count  Dtype  
---  ------                                       --------------  -----  
 0   puntuacion                                   15581 non-null  int64  
 1   age                                          15581 non-null  int64  
 2   Agricultura ecológica                        15581 non-null  int32  
 3   Arte                                         15581 non-null  int32  
 4   Asador                                       15581 non-null  int32  
 5   Bertsolarismo                                15581 non-null  int32  
 6   Bodegas                                      15581 non-null  int32  
 7   Ciencias naturales                           15581 non-null  int32  
 8   Cine y audiovisuales                         15581 non-null  int32  
 9   Concierto y Festival                         15581 non-null  int32  
 10

In [31]:
df_reviews_total.to_csv("ML/data/reviews_gastro.csv", index=False)

# MAQUINE LEARNING

# MODELADO


In [32]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, HistGradientBoostingRegressor
from sklearn.dummy import DummyRegressor

# separo target y features
X = df_reviews_total.drop(columns=["puntuacion"])
y = df_reviews_total["puntuacion"]

# nos quedamos solo con columnas numéricas y booleanas para el modelado
X = X.select_dtypes(include=["number", "bool"]).copy()

# pasamos booleanos a enteros para que todos los modelos trabajen con una matriz numérica homogénea
bool_cols = X.select_dtypes(include=["bool"]).columns
X[bool_cols] = X[bool_cols].astype(int)

# split 80/20
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)
print("Columnas con NaN en X:", int(X.isna().any().sum()))


X shape: (15581, 61)
y shape: (15581,)
Train shape: (12464, 61)
Test shape: (3117, 61)
Columnas con NaN en X: 0


In [33]:
models = {
    "Dummy": DummyRegressor(strategy="mean"),
    "Ridge": Ridge(alpha=1.0, random_state=42),
    "RandomForest": RandomForestRegressor(
        n_estimators=200,
        random_state=42,
        n_jobs=-1
    ),
    "GradientBoosting": GradientBoostingRegressor(random_state=42),
    "HistGradientBoosting": HistGradientBoostingRegressor(random_state=42)
}

results = []
trained_models = {}

for name, estimator in models.items():
    pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", estimator)
    ])
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)

    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)

    results.append({
        "modelo": name,
        "RMSE": rmse,
        "MAE": mae,
        "R2": r2
    })
    trained_models[name] = pipeline

results_df = pd.DataFrame(results).sort_values(by="RMSE").reset_index(drop=True)
best_model_name = results_df.loc[0, "modelo"]
best_model = trained_models[best_model_name]

print("Comparativa de modelos ordenada por RMSE:")
results_df


Comparativa de modelos ordenada por RMSE:


,modelo,RMSE,MAE,R2
0,HistGradientBoosting,0.990359,0.749277,0.062086
1,GradientBoosting,0.991826,0.747279,0.059305
2,Ridge,0.993962,0.756484,0.055248
3,Dummy,1.022782,0.765350,-0.000333
4,RandomForest,1.039697,0.810241,-0.033693


# Patrimonio

In [34]:
df_patrimonio = pd.read_csv("data/patrimonio.csv")
df_patrimonio

,Unnamed: 0,Nombre,Descripción,Tipo de lugar,Dirección,Municipio,Provincia,Postal Code,Email,Teléfono,...,Capacidad,Tienda,Tipo de Cultura,lat,lon,valoracion,num_resenas,url_imagen,Active,Patrocinado
0,0,Museo Bibat,"Ubicado en el Casco Antiguo de Vitoria, consta...",Museo,"Cuchillería, 54. Junto al Palacio de Bendaña.",Vitoria,Álava,10010,bibat@araba.eus,945 203 700,...,0,1,Otros,42.849309,-2.671176,4.4,2420.0,https://places.googleapis.com/v1/places/ChIJd6...,1,0
1,1,Museo de Faroles de la Cofradía de la Virgen B...,Este original museo alberga las 267 piezas de ...,Museo,"Zapatería, 33",Vitoria,Álava,10010,info@cofradiavirgenblanca.com,945 277 077,...,1000,0,Otros,42.848097,-2.674140,4.7,2500.0,https://places.googleapis.com/v1/places/ChIJN4...,1,0
2,2,Museo de Ciencias Naturales de Álava,El Museo de Ciencias Naturales de Álava cuenta...,Museo,"Fundadora de las Siervas de Jesús, 24",Vitoria,Álava,10010,mcna@araba.eus,945 181 924,...,1000,0,Ciencias Naturales,42.850290,-2.674625,4.6,7700.0,https://places.googleapis.com/v1/places/ChIJHQ...,1,0
3,3,Artium museoa,El/la visitante podrá encontrar en exposición ...,Museo,"Francia, 24",Vitoria,Álava,10020,museo@artium.eus,945 209 020,...,28920,1,Arte,42.849426,-2.668021,4.1,26440.0,https://places.googleapis.com/v1/places/ChIJfz...,1,0
4,4,Museo de Armería,El Museo de Armería ofrece al visitante una ex...,Museo,"Paseo de Fray Francisco de Vitoria, 3",Vitoria,Álava,10010,museoarmeria@araba.eus,945 181 925,...,920,0,Otros,42.841347,-2.678544,4.7,7290.0,https://places.googleapis.com/v1/places/ChIJ2W...,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
572,572,Jardín Botánico Berrizjauregi,El Jardín Botánico Berrizjauregi está ubicado ...,Patrimonio cultural,"Iturritza Kalea, 51",Berriz,Vizcaya,48240,NaN,946 824 036,...,0,0,Patrimonio Cultural,43.174847,-2.577543,4.8,120.0,https://places.googleapis.com/v1/places/ChIJFT...,1,0
573,573,Galería J. M. Lumbreras,Esta galería posee tres salas de exposición y ...,Museo,c/ Henao 3,Bilbao,Vizcaya,48009,galeria@galerialumbreras.com,944 244 545,...,0,0,Patrimonio Cultural,43.263960,-2.929496,4.7,760.0,https://places.googleapis.com/v1/places/ChIJyS...,1,0
574,574,Castillo de Portilla,"Está ubicado en Portilla, concejo del municipi...",Patrimonio cultural,Portilla/Zabalate,Zambrana,Álava,1212,NaN,NaN,...,0,0,Patrimonio Cultural,42.670370,-2.833645,NaN,NaN,NaN,1,0
575,575,ABAO Bilbao Ópera,ABAO Bilbao Ópera es una institución cultural ...,Museo,"José María Olavarri, 2",Bilbao,Vizcaya,48001,abao@abao.org,NaN,...,0,0,Patrimonio Cultural,43.260347,-2.927401,4.4,780.0,https://places.googleapis.com/v1/places/ChIJu-...,1,0


In [35]:
# renombramos la Unnamed: 0 como patrimonio_id
df_patrimonio.rename(columns={"Unnamed: 0": "patrimonio_id"}, inplace=True)

# eliminamos la columna "Nombre", "Descripción", "Dirección", "Municipio", "Postal Code", "Email", "Teléfono", "WEB", "URL amigable", "lat", "lon", "url_imagen", "Active", "Patrocinado"
df_patrimonio.drop(columns=["Nombre", "Descripción", "Dirección", "Municipio", "Postal Code", "Email", "Teléfono", "WEB", "URL amigable", "lat", "lon", "url_imagen", "Active", "Patrocinado"], inplace=True)

In [36]:
df_patrimonio

,patrimonio_id,Tipo de lugar,Provincia,Visita Guiada,Capacidad,Tienda,Tipo de Cultura,valoracion,num_resenas
0,0,Museo,Álava,1,0,1,Otros,4.4,2420.0
1,1,Museo,Álava,1,1000,0,Otros,4.7,2500.0
2,2,Museo,Álava,1,1000,0,Ciencias Naturales,4.6,7700.0
3,3,Museo,Álava,1,28920,1,Arte,4.1,26440.0
4,4,Museo,Álava,1,920,0,Otros,4.7,7290.0
...,...,...,...,...,...,...,...,...,...
572,572,Patrimonio cultural,Vizcaya,0,0,0,Patrimonio Cultural,4.8,120.0
573,573,Museo,Vizcaya,0,0,0,Patrimonio Cultural,4.7,760.0
574,574,Patrimonio cultural,Álava,0,0,0,Patrimonio Cultural,NaN,NaN
575,575,Museo,Vizcaya,0,0,0,Patrimonio Cultural,4.4,780.0


In [37]:
# analizamos los distintos valores de la columna "Tipo de lugar"
print("Valores únicos en la columna 'Tipo de lugar':", df_patrimonio["Tipo de lugar"].unique())

Valores únicos en la columna 'Tipo de lugar': ['Museo' 'Patrimonio cultural']


In [38]:
# analizamos los distintos valores de la columna "Provincia"
print("Valores únicos en la columna 'Tipo de lugar':", df_patrimonio["Provincia"].unique())

Valores únicos en la columna 'Tipo de lugar': ['Álava' 'Guipúzcoa' 'Vizcaya']


In [39]:
# analizamos los distintos valores de la columna "Tipo de Cultura"
print("Valores únicos en la columna 'Tipo de Cultura':", df_patrimonio["Tipo de Cultura"].unique())

Valores únicos en la columna 'Tipo de Cultura': ['Otros' 'Ciencias Naturales' 'Arte' 'Gastronomía' 'Religión' 'Etnografía'
 'Historia' 'Artesanía' 'Marítimo' 'Patrimonio Cultural']


In [40]:
# hacemos un one-hot de "Tipo de lugar", "Provincia" y "Tipo de Cultura"
df_patrimonio = pd.get_dummies(df_patrimonio, columns=["Tipo de lugar", "Provincia", "Tipo de Cultura"])

In [41]:
# convertimos a booleanas las columnas de "Visita Guiada", "Tienda"
df_patrimonio["Visita Guiada"] = df_patrimonio["Visita Guiada"].astype(bool)
df_patrimonio["Tienda"] = df_patrimonio["Tienda"].astype(bool)

In [42]:
df_patrimonio.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 577 entries, 0 to 576
Data columns (total 21 columns):
 #   Column                               Non-Null Count  Dtype  
---  ------                               --------------  -----  
 0   patrimonio_id                        577 non-null    int64  
 1   Visita Guiada                        577 non-null    bool   
 2   Capacidad                            577 non-null    int64  
 3   Tienda                               577 non-null    bool   
 4   valoracion                           510 non-null    float64
 5   num_resenas                          510 non-null    float64
 6   Tipo de lugar_Museo                  577 non-null    bool   
 7   Tipo de lugar_Patrimonio cultural    577 non-null    bool   
 8   Provincia_Guipúzcoa                  577 non-null    bool   
 9   Provincia_Vizcaya                    577 non-null    bool   
 10  Provincia_Álava                      577 non-null    bool   
 11  Tipo de Cultura_Arte            

In [43]:
df_patrimonio

,patrimonio_id,Visita Guiada,Capacidad,Tienda,valoracion,num_resenas,Tipo de lugar_Museo,Tipo de lugar_Patrimonio cultural,Provincia_Guipúzcoa,Provincia_Vizcaya,...,Tipo de Cultura_Arte,Tipo de Cultura_Artesanía,Tipo de Cultura_Ciencias Naturales,Tipo de Cultura_Etnografía,Tipo de Cultura_Gastronomía,Tipo de Cultura_Historia,Tipo de Cultura_Marítimo,Tipo de Cultura_Otros,Tipo de Cultura_Patrimonio Cultural,Tipo de Cultura_Religión
0,0,True,0,True,4.4,2420.0,True,False,False,False,...,False,False,False,False,False,False,False,True,False,False
1,1,True,1000,False,4.7,2500.0,True,False,False,False,...,False,False,False,False,False,False,False,True,False,False
2,2,True,1000,False,4.6,7700.0,True,False,False,False,...,False,False,True,False,False,False,False,False,False,False
3,3,True,28920,True,4.1,26440.0,True,False,False,False,...,True,False,False,False,False,False,False,False,False,False
4,4,True,920,False,4.7,7290.0,True,False,False,False,...,False,False,False,False,False,False,False,True,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
572,572,False,0,False,4.8,120.0,False,True,False,True,...,False,False,False,False,False,False,False,False,True,False
573,573,False,0,False,4.7,760.0,True,False,False,True,...,False,False,False,False,False,False,False,False,True,False
574,574,False,0,False,NaN,NaN,False,True,False,False,...,False,False,False,False,False,False,False,False,True,False
575,575,False,0,False,4.4,780.0,True,False,False,True,...,False,False,False,False,False,False,False,False,True,False


# RESEÑAS PATRIMONIO

In [44]:
df_reviews_patrimonio = pd.read_csv("data/culture_reviews_claude.csv")
df_reviews_patrimonio

,user_id,culture_id,puntuacion,texto,created_at
0,1467,1,4,NaN,2024-10-08 00:00:00
1,1200,1,3,NaN,2024-04-14 00:00:00
2,306,1,4,"Muy buena visita, lo recomiendo.",2025-08-27 00:00:00
3,115,1,4,NaN,2024-08-11 00:00:00
4,1738,1,5,Personal amable y exposición excelente.,2024-07-22 00:00:00
...,...,...,...,...,...
7568,930,577,4,"Vale la pena, aunque esperaba algo más de algu...",2024-09-10 00:00:00
7569,334,577,5,Personal amable y exposición excelente.,2024-06-16 00:00:00
7570,35,577,4,"Vale la pena, aunque esperaba algo más de algu...",2024-04-06 00:00:00
7571,1221,577,4,Interesante y bien presentado.,2024-12-09 00:00:00


In [45]:
# eliminamos las columnas "id", "texto", "created_at"
df_reviews_patrimonio.drop(columns=["texto", "created_at"], inplace=True)
df_reviews_patrimonio

,user_id,culture_id,puntuacion
0,1467,1,4
1,1200,1,3
2,306,1,4
3,115,1,4
4,1738,1,5
...,...,...,...
7568,930,577,4
7569,334,577,5
7570,35,577,4
7571,1221,577,4


In [46]:
df_reviews_patrimonio_total = (
    df_reviews_patrimonio
    .merge(df_user_total, left_on="user_id", right_on="id_user", how="left")
    .merge(df_patrimonio, left_on="culture_id", right_on="patrimonio_id", how="left")
)

# comprobación rápida
print("Filas df_reviews:", len(df_reviews))
print("Filas df_reviews_patrimonio_total:", len(df_reviews_patrimonio_total))
df_reviews_patrimonio_total.head()

Filas df_reviews: 15581
Filas df_reviews_patrimonio_total: 7573


,user_id,culture_id,puntuacion,id_user,age,Agricultura ecológica,Arte,Asador,Bertsolarismo,Bodegas,...,Tipo de Cultura_Arte,Tipo de Cultura_Artesanía,Tipo de Cultura_Ciencias Naturales,Tipo de Cultura_Etnografía,Tipo de Cultura_Gastronomía,Tipo de Cultura_Historia,Tipo de Cultura_Marítimo,Tipo de Cultura_Otros,Tipo de Cultura_Patrimonio Cultural,Tipo de Cultura_Religión
0,1467,1,4,1467,59,0,0,1,0,1,...,False,False,False,False,False,False,False,True,False,False
1,1200,1,3,1200,58,0,1,1,0,1,...,False,False,False,False,False,False,False,True,False,False
2,306,1,4,306,55,0,1,1,0,1,...,False,False,False,False,False,False,False,True,False,False
3,115,1,4,115,65,0,1,1,0,1,...,False,False,False,False,False,False,False,True,False,False
4,1738,1,5,1738,49,0,1,1,0,1,...,False,False,False,False,False,False,False,True,False,False


In [47]:
df_reviews_patrimonio_total = df_reviews_patrimonio_total.drop(columns=["user_id", "id_user", "culture_id", "patrimonio_id"])

In [48]:
df_reviews_patrimonio_total

,puntuacion,age,Agricultura ecológica,Arte,Asador,Bertsolarismo,Bodegas,Ciencias naturales,Cine y audiovisuales,Concierto y Festival,...,Tipo de Cultura_Arte,Tipo de Cultura_Artesanía,Tipo de Cultura_Ciencias Naturales,Tipo de Cultura_Etnografía,Tipo de Cultura_Gastronomía,Tipo de Cultura_Historia,Tipo de Cultura_Marítimo,Tipo de Cultura_Otros,Tipo de Cultura_Patrimonio Cultural,Tipo de Cultura_Religión
0,4,59,0,0,1,0,1,0,0,0,...,False,False,False,False,False,False,False,True,False,False
1,3,58,0,1,1,0,1,0,0,0,...,False,False,False,False,False,False,False,True,False,False
2,4,55,0,1,1,0,1,0,0,0,...,False,False,False,False,False,False,False,True,False,False
3,4,65,0,1,1,0,1,0,0,0,...,False,False,False,False,False,False,False,True,False,False
4,5,49,0,1,1,0,1,0,0,0,...,False,False,False,False,False,False,False,True,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7568,4,46,0,1,0,0,1,0,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7569,5,57,0,0,0,0,0,0,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7570,4,54,0,0,1,0,0,0,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7571,4,46,0,1,1,0,1,0,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [49]:
df_reviews_patrimonio_total.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7573 entries, 0 to 7572
Data columns (total 54 columns):
 #   Column                                       Non-Null Count  Dtype  
---  ------                                       --------------  -----  
 0   puntuacion                                   7573 non-null   int64  
 1   age                                          7573 non-null   int64  
 2   Agricultura ecológica                        7573 non-null   int32  
 3   Arte                                         7573 non-null   int32  
 4   Asador                                       7573 non-null   int32  
 5   Bertsolarismo                                7573 non-null   int32  
 6   Bodegas                                      7573 non-null   int32  
 7   Ciencias naturales                           7573 non-null   int32  
 8   Cine y audiovisuales                         7573 non-null   int32  
 9   Concierto y Festival                         7573 non-null   int32  
 10  

In [50]:
df_reviews_patrimonio_total.to_csv("ML/data/df_reviews_patrimonio_total.csv", index=False)

# MACHINE 

In [51]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, HistGradientBoostingRegressor
from sklearn.dummy import DummyRegressor

# separo target y features
X = df_reviews_patrimonio_total.drop(columns=["puntuacion"])
y = df_reviews_patrimonio_total["puntuacion"]

# nos quedamos solo con columnas numéricas y booleanas para el modelado
X = X.select_dtypes(include=["number", "bool"]).copy()

# pasamos booleanos a enteros para que todos los modelos trabajen con una matriz numérica homogénea
bool_cols = X.select_dtypes(include=["bool"]).columns
X[bool_cols] = X[bool_cols].astype(int)

# split 80/20
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)
print("Columnas con NaN en X:", int(X.isna().any().sum()))


X shape: (7573, 36)
y shape: (7573,)
Train shape: (6058, 36)
Test shape: (1515, 36)
Columnas con NaN en X: 3


In [52]:
models = {
    "Dummy": DummyRegressor(strategy="mean"),
    "Ridge": Ridge(alpha=1.0, random_state=42),
    "RandomForest": RandomForestRegressor(
        n_estimators=200,
        random_state=42,
        n_jobs=-1
    ),
    "GradientBoosting": GradientBoostingRegressor(random_state=42),
    "HistGradientBoosting": HistGradientBoostingRegressor(random_state=42)
}

results = []
trained_models = {}

for name, estimator in models.items():
    pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", estimator)
    ])
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)

    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)

    results.append({
        "modelo": name,
        "RMSE": rmse,
        "MAE": mae,
        "R2": r2
    })
    trained_models[name] = pipeline

results_df = pd.DataFrame(results).sort_values(by="RMSE").reset_index(drop=True)
best_model_name = results_df.loc[0, "modelo"]
best_model = trained_models[best_model_name]

print("Comparativa de modelos ordenada por RMSE:")
results_df


Comparativa de modelos ordenada por RMSE:


,modelo,RMSE,MAE,R2
0,GradientBoosting,0.854522,0.628257,0.009564
1,Ridge,0.858463,0.633959,0.000409
2,Dummy,0.859064,0.579751,-0.000993
3,HistGradientBoosting,0.862958,0.645124,-0.010088
4,RandomForest,0.886831,0.671768,-0.066746


In [53]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, HistGradientBoostingRegressor
from sklearn.dummy import DummyRegressor

# ── 1. Cargar los datos nuevos ────────────────────────────────────────────────
reviews      = pd.read_csv("data/culture_reviews_claude.csv")          # el nuevo
patrimonio   = pd.read_csv("data/patrimonio.csv")
usuarios     = pd.read_csv("data/usuarios.csv")
int_usuarios = pd.read_csv("data/intereses_usuarios.csv")

patrimonio["culture_id"] = patrimonio["Unnamed: 0"] + 1

# ── 2. Construir feature de afinidad usuario-lugar ────────────────────────────
AFINIDAD = {
    25: ["Historia"],
    26: ["Ciencias Naturales"],
    27: ["Arte", "Artesanía"],
    28: ["Etnografía"],
    29: ["Patrimonio Cultural", "Religión", "Marítimo", "Otros"],
    11: ["Arte", "Otros"],
}

user_interests = (
    int_usuarios.groupby("id_user")["id_interes"]
    .apply(set).to_dict()
)

def calcular_afinidad(uid, tipo_cultura):
    intereses_u = user_interests.get(uid, set())
    score = sum(
        1 for iid, tipos in AFINIDAD.items()
        if iid in intereses_u and tipo_cultura in tipos
    )
    return min(score, 3) / 3.0

# ── 3. Pivot de intereses por usuario (one-hot) ───────────────────────────────
INTERESES_CULTURALES = [25, 26, 27, 28, 29]
for iid in INTERESES_CULTURALES:
    col = f"interes_{iid}"
    usuarios_con = set(
        int_usuarios[int_usuarios["id_interes"] == iid]["id_user"]
    )
    reviews[col] = reviews["user_id"].apply(lambda u: int(u in usuarios_con))

# ── 4. Join reviews ← usuarios ← patrimonio ──────────────────────────────────
df = reviews.merge(
    usuarios[["id_user", "age", "sexo", "municipality_id"]],
    left_on="user_id", right_on="id_user", how="left"
).merge(
    patrimonio[[
        "culture_id", "Tipo de Cultura", "Tipo de lugar",
        "valoracion", "num_resenas", "Capacidad", "Tienda",
        "Visita Guiada", "Provincia"
    ]],
    on="culture_id", how="left"
)

# ── 5. Feature de afinidad cruzada (LA MÁS IMPORTANTE) ───────────────────────
df["afinidad"] = df.apply(
    lambda r: calcular_afinidad(r["user_id"], r["Tipo de Cultura"]),
    axis=1
)

# ── 6. Codificar categóricas ──────────────────────────────────────────────────
df = pd.get_dummies(df, columns=["sexo", "Tipo de Cultura", "Tipo de lugar", "Provincia"])

# ── 7. Features finales ───────────────────────────────────────────────────────
drop_cols = ["user_id", "culture_id", "texto", "created_at",
             "puntuacion", "id_user", "municipality_id"]
X = df.drop(columns=[c for c in drop_cols if c in df.columns])
X = X.select_dtypes(include=["number", "bool"]).copy()
bool_cols = X.select_dtypes(include=["bool"]).columns
X[bool_cols] = X[bool_cols].astype(int)
y = df["puntuacion"]

print("Features:", X.shape[1])
print("Columnas clave presentes:")
print([c for c in X.columns if "afinidad" in c or "interes" in c or "valoracion" in c])

Features: 30
Columnas clave presentes:
['interes_25', 'interes_26', 'interes_27', 'interes_28', 'interes_29', 'valoracion', 'afinidad']
